In [ ]:
import os
import pandas as pd
import scanpy as sc
from sklearn.manifold import TSNE

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

import plotnine as gg

In [ ]:
SAVE_DIR = "/workspace/experiments/01022026_multimodal"
spacer_predictability = pd.read_csv(os.path.join(SAVE_DIR, "spacer_predictability.csv"))
gene_predictability = pd.read_csv(os.path.join(SAVE_DIR, "gene_predictability.csv"))

In [ ]:
adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata.X = adata.layers["reads"].copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata.obs["transcript_UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["transcript_UMAP2"] = adata.obsm["X_umap"][:, 1]

adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

In [ ]:
adata_case

### RNA polymerase genes

In *Escherichia coli*, the genes **rpoA, rpoB, rpoC, rpoD, rpoE, rpoH, rpoN, rpoS,** and **rpoZ** encode the structural subunits and specificity factors (sigma factors) of the RNA Polymerase (RNAP), the central engine of gene expression.

These genes can be categorized into two groups: those encoding the **Core Enzyme** (the machinery) and those encoding **Sigma Factors** (the guides that tell the machinery where to start).

### 1. The Core Enzyme Genes (*rpoA, rpoB, rpoC, rpoZ*)
The core RNA polymerase is responsible for the actual catalysis of RNA synthesis (transcription elongation). It has the subunit composition **$\alpha_2\beta\beta'\omega$**.

#### **rpoA** ($\alpha$ subunit)
*   **Role:** The *rpoA* gene encodes the $\alpha$ subunit. Two copies of $\alpha$ form a dimer ($\alpha_2$) that serves as the scaffold for RNAP assembly.
*   **Structure & Function:**
    *   **N-terminal domain ($\alpha$NTD):** Essential for dimerization and recruitment of the $\beta$ subunit.
    *   **C-terminal domain ($\alpha$CTD):** Connected by a flexible linker, this domain binds to "UP elements" (DNA sequences upstream of the -35 promoter region) to enhance transcription initiation, particularly at rRNA promoters. It is also the primary target for transcriptional activators (e.g., CRP, CAP) to interact with RNAP.

#### **rpoB** ($\beta$ subunit)
*   **Role:** Encodes the $\beta$ subunit, the second largest subunit.
*   **Structure & Function:** Forms the catalytic center of the enzyme along with $\beta'$. It contains the ribonucleoside triphosphate (NTP) entry channel and the binding site for the antibiotic **rifampicin**. It handles the initiation and elongation steps of nucleotide addition.

#### **rpoC** ($\beta'$ subunit)
*   **Role:** Encodes the $\beta'$ subunit, the largest subunit.
*   **Structure & Function:** Forms the "pincer" or "crab claw" structure that grips the DNA template (the DNA clamp). It contains the active site magnesium ions ($Mg^{2+}$) essential for catalysis. It works tightly with $\beta$ to form the active site channel.

#### **rpoZ** ($\omega$ subunit)
*   **Role:** Encodes the $\omega$ subunit, the smallest component.
*   **Structure & Function:** Historically considered non-essential, literature now emphasizes its role in **structural stability**. It acts like a latch, securing the N- and C- termini of the $\beta'$ subunit to properly fold it and facilitate its assembly into the complex.
*   **Interaction:** It recruits the chaperone **GroEL** to assist in RNAP assembly. It is also functionally linked to the stringent response; *rpoZ* mutants show defects in the synthesis of ppGpp (the alarmone) because the $\omega$ subunit aids the function of RelA (ppGpp synthase).

---

### 2. The Sigma Factor Genes (*rpoD, rpoS, rpoH, rpoE, rpoN*)
The core enzyme cannot recognize promoter sequences on its own. It requires a sigma ($\sigma$) factor to form the **Holoenzyme** ($\alpha_2\beta\beta'\omega\sigma$), which directs the polymerase to specific gene promoters.

#### **rpoD** ($\sigma^{70}$ - The Housekeeper)
*   **Role:** The primary sigma factor responsible for transcribing most genes during exponential growth (housekeeping genes), including metabolic enzymes and ribosomal components.
*   **Mechanism:** Recognizes the standard -10 (TATAAT) and -35 (TTGACA) promoter consensus sequences.

#### **rpoS** ($\sigma^{38}$ / $\sigma^S$ - The General Stress Regulator)
*   **Role:** The master regulator of the **stationary phase** and general stress response.
*   **Function:** Activated during nutrient starvation, osmotic stress, or acidic conditions. It directs RNAP to genes necessary for survival, such as those for cell wall modification and stress resistance (e.g., catalase).
*   **Regulation:** *rpoS* is heavily regulated at the level of translation and protein stability (by the adaptor protein RssB and ClpXP protease) to prevent its activity during exponential growth.

#### **rpoH** ($\sigma^{32}$ - Heat Shock)
*   **Role:** Controls the cytoplasmic **heat shock response**.
*   **Function:** Transcriptionally activates genes encoding chaperones (e.g., DnaK, GroEL) and proteases (e.g., Lon) that manage protein folding and degrade damaged proteins.
*   **Regulation:** Under normal conditions, RpoH is rapidly degraded or sequestered by DnaK. Upon heat stress, unfolded proteins titrate DnaK away, releasing RpoH to drive transcription.

#### **rpoE** ($\sigma^{24}$ - Envelope Stress)
*   **Role:** Responds to **extracytoplasmic (envelope) stress**, specifically misfolded proteins in the periplasm or outer membrane.
*   **Function:** Activates genes for periplasmic chaperones and factors involved in outer membrane porin assembly. It is essential for viability at high temperatures.
*   **Regulation:** It is normally sequestered at the inner membrane by the anti-sigma factor **RseA**. Stress causes the degradation of RseA, releasing RpoE into the cytoplasm.

#### **rpoN** ($\sigma^{54}$ - Nitrogen/Alternative)
*   **Role:** Regulates nitrogen metabolism and other specific functions (like motility and biofilm formation).
*   **Mechanism:** Unique among sigma factors, RpoN-RNAP forms a stable closed complex that cannot initiate transcription spontaneously. It requires an **enhancer-binding protein** (e.g., NtrC) and ATP hydrolysis to melt the DNA and open the transcription bubble.

---

### 3. Interactions and Regulatory Networks

The interplay between these genes defines the bacterium's physiological state.

#### **A. Assembly Pathway**
The physical interaction of the gene products follows a strict order to form the operational enzyme:
1.  **2 x RpoA ($\alpha$)** dimerize ($\alpha_2$).
2.  $\alpha_2$ binds **RpoB ($\beta$)** $\rightarrow$ $\alpha_2\beta$.
3.  **RpoZ ($\omega$)** facilitates the folding and binding of **RpoC ($\beta'$)** to the complex $\rightarrow$ **Core Enzyme ($\alpha_2\beta\beta'\omega$)**.
4.  The Core binds one of the **Sigma factors** to form the **Holoenzyme**.

#### **B. The "Sigma Cycle" and Competition**
There is a limited pool of Core RNAP in the cell, and sigma factors must compete to bind it. This competition is a major regulatory mechanism:
*   **Exponential Growth:** **RpoD** has the highest affinity and abundance, so it dominates, driving growth genes.
*   **Stress/Starvation:** The small molecule **ppGpp** (guanosine tetraphosphate) accumulates. ppGpp destabilizes RpoD-promoter open complexes and favors the binding of alternative sigmas like **RpoS**, **RpoH**, and **RpoE**.
*   **Anti-Sigma Factors:** Proteins like Rsd (anti-RpoD) can sequester RpoD during stationary phase, further allowing RpoS to capture the Core RNAP.

#### **C. Transcriptional Cascades**
The sigma factors regulate each other in a hierarchical network:
*   **RpoD $\rightarrow$ RpoS/RpoE/RpoH:** The "housekeeping" sigma (RpoD) transcribes the basal levels of the stress sigmas (*rpoS*, *rpoE*, *rpoH*).
*   **RpoE $\rightarrow$ RpoH:** Under extreme heat stress, **RpoE** recognizes a specific promoter upstream of the *rpoH* gene. This ensures that if the cell envelope is melting, the cytoplasmic heat shock response (RpoH) is also boosted.
*   **RpoS Self-Reinforcement:** RpoS can drive the expression of proteins that stabilize itself, creating a positive feedback loop during deep stationary phase.

In [ ]:
adata_sub = adata_case.obs.loc[lambda x: x["target"].astype(str).str.startswith("rpo")].copy()
adata_sub["target"] = adata_sub["target"].astype(str)

(
    gg.ggplot(
        adata_case.obs,
    )
    + gg.geom_point(
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"),
        size=0.5,
    )
    + gg.geom_point(
        adata_sub,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="target"),
        size=3,
    )
    + gg.theme_minimal()
    + gg.theme(figure_size=(12, 10))
)

To understand why *rpoZ* might cluster differently than the other polymerase genes, we first need to clear up the definitions of the different RNA types.

### Part 1: mRNA, tRNA, rRNA — The Difference

To answer your specific question: **No, all three types are not produced for a single given gene.**

A specific gene in the DNA generally codes for **one** specific type of RNA. The type of RNA produced depends on the function of that gene.

1.  **mRNA (Messenger RNA) — "The Script"**
    *   **Function:** It carries the genetic code from the DNA to the ribosome. It acts as the instruction manual for building a specific protein.
    *   **Life cycle:** It is very unstable and short-lived. It is made, translated into protein, and then degraded quickly.
    *   **Genes:** Most genes in *E. coli* (like *rpoA*, *lacZ*, *dnaK*) code for mRNAs.

2.  **rRNA (Ribosomal RNA) — "The Factory"**
    *   **Function:** It does *not* code for a protein. Instead, the RNA itself folds up to become the physical structure and mechanical engine of the ribosome (the machine that builds proteins).
    *   **Life cycle:** Very stable.
    *   **Genes:** In *E. coli*, these are the *rrn* operons (e.g., *rrnB*).

3.  **tRNA (Transfer RNA) — "The Truck"**
    *   **Function:** It does *not* code for a protein. It acts as an adaptor. It picks up a specific amino acid and ferries it to the ribosome to be added to the growing protein chain.
    *   **Life cycle:** Very stable.
    *   **Genes:** These are called *trn* genes (e.g., *trpT* encodes the tRNA for Tryptophan).

**Crucial Term: "tRNA-Charging"**
The genes you mentioned ("tRNA-charging genes") are actually **protein-coding genes** (mRNA). They code for enzymes called **Aminoacyl-tRNA Synthetases**. These enzymes are the "loaders"—they grab an empty tRNA and attach the correct amino acid to it.

---

### Part 2: Is it expected that *rpoZ* clusters with tRNA-charging genes?

**Yes, this is biologically expected,** provided you are looking at functional clustering (co-expression or regulatory networks) rather than just physical proximity on the chromosome.

Here is why *rpoZ* behaves differently than *rpoA*, *rpoB*, and *rpoC*:

#### 1. The "Growth vs. Control" Separation
*   **The Core (*rpoA, rpoB, rpoC*):** These genes build the heavy machinery of the polymerase. Their expression is tightly linked to **Ribosomes**. When the cell grows fast, it needs more Polymerase and more Ribosomes. Therefore, *rpoB* and *rpoC* are actually found in a physical operon *with* ribosomal protein genes. They cluster with "Growth" machinery.
*   **The Latch (*rpoZ*):** The $\omega$ subunit (RpoZ) is not strictly required for the enzyme to fire. Instead, it ensures the enzyme folds correctly and, crucially, **responds to stress**. It clusters with "Regulation/Sensing" machinery.

#### 2. The Functional Link: The Stringent Response
The logic connecting *rpoZ* to tRNA-charging genes lies in the **Stringent Response** (the starvation response we discussed earlier).

*   **The Sensor:** The cell knows it is starving when **tRNA-charging** fails (i.e., there are uncharged tRNAs because there are no amino acids).
*   **The Signal:** This "uncharged tRNA" state triggers RelA to produce **ppGpp**.
*   **The Responder:** As established, **RpoZ ($\omega$)** is required for ppGpp to bind the Polymerase and for RelA to function.

Therefore, the cell coordinates the expression of **tRNA-charging enzymes** (which maintain amino acid levels) with **RpoZ** (which helps the cell survive when those levels drop). They are part of the same "homeostatic" network—monitoring and maintaining the nutrient status of the cell.

#### 3. Genomic Context
In the *E. coli* genome, *rpoZ* is located in an operon immediately adjacent to **spoT**.
*   **spoT** is the enzyme responsible for **degrading and synthesizing ppGpp**.
*   This physical clustering confirms that the primary role of *rpoZ* is tied to the management of the alarmone (ppGpp) and metabolic sensing, rather than just being a "brick" in the polymerase wall like *rpoA*.

**Summary:**
While *rpoA/B/C* cluster with the "Construction Crew" (Ribosomes) to drive growth, *rpoZ* clusters with the "Quality Control & Supply Chain Managers" (tRNA-charging enzymes and ppGpp regulators) to manage resources.

In [ ]:
", ".join(adata.var[adata.var.index.str.startswith("rpo")].index)

In [ ]:
is_control_fn = lambda x: x["gene"].str.contains("Control")
is_case_fn = lambda x: x["gene"].str.startswith("rpo")
is_relevant_fn = lambda x: is_case_fn(x) | is_control_fn(x)


adata_subset = adata[adata.obs.loc[is_relevant_fn].index].copy()
adata_subset.obs["target_"] = adata_subset.obs["target"].astype(str)
adata_subset.obs.loc[lambda x: x["target_"].str.contains("Control"), "target_"] = "Control"

In [ ]:
adata_subset_fix = adata_subset[
    adata_subset.obs.loc[lambda x: ~x["target_"].isin(["rpoC"])].index
].copy()
sc.pl.dotplot(
    adata_subset_fix,
    var_names=adata_subset_fix.var_names[adata_subset_fix.var.index.str.startswith("rpo")],
    groupby="target_",
)

In [ ]:
adata_ctrl_exp = adata_subset_fix[
    adata_subset_fix.obs.loc[lambda x: x["target_"] == "nontargeting"].index
].copy()
for target in ["rpoA", "rpoB", "rpoC", "rpoD", "rpoE", "rpoH", "rpoN", "rpoS", "rpoZ"]:
    baseline_expression = adata_ctrl_exp[:, target].layers["cp10k"].mean()
    adata_pert = adata_subset_fix[
        adata_subset_fix.obs.loc[lambda x: x["target_"] == target].index
    ].copy()
    if adata_pert.shape[0] == 0:
        continue
    pert_expression = adata_pert[:, target].layers["cp10k"].mean()
    print(target)
    print("baseline", baseline_expression)
    print("pert", pert_expression)
    print(f"{pert_expression / baseline_expression:.2f}")
    print()

In [ ]:
sc.tl.rank_genes_groups(adata_subset_fix, groupby="target_", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata_subset_fix, n_genes=5)

In [ ]:
sc.tl.rank_genes_groups(
    adata_subset_fix, groupby="target_", method="wilcoxon", reference="nontargeting"
)
for group in adata_subset_fix.obs["target_"].unique():
    if group == "nontargeting":
        continue
    print(group)
    n_obs = adata_subset_fix.obs.loc[lambda x: x["target_"] == group].shape[0]
    print(f"{n_obs} observations")
    display(sc.get.rank_genes_groups_df(adata_subset_fix, group=group).query("pvals_adj < 0.1"))
    print()

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_subset_fix, n_genes=5, groupby="target_", standard_scale="var"
)

In [ ]:
sc.pp.highly_variable_genes(adata_subset_fix, n_top_genes=500)

In [ ]:
sc.pp.pca(adata_subset_fix, use_highly_variable=True)
sc.pl.pca(adata_subset_fix, color=["target_"])
sc.pp.neighbors(adata_subset_fix)
sc.tl.umap(adata_subset_fix)
sc.pl.umap(
    adata_subset_fix,
    color=["target_"],
)

In [ ]:
from sklearn.decomposition import KernelPCA

X = adata_subset_fix[:, adata_subset_fix.var.highly_variable].X.toarray()

kernel_pca = KernelPCA(
    n_components=2, kernel="cosine", gamma=None, fit_inverse_transform=True, alpha=0.1
)
X_kpca = kernel_pca.fit_transform(X)
X_kpca.shape

rep_df = pd.DataFrame(X_kpca, index=adata_subset_fix.obs_names, columns=["PC1", "PC2"]).assign(
    target=adata_subset_fix.obs["target_"]
)
(
    gg.ggplot(rep_df)
    + gg.geom_point(gg.aes(x="PC1", y="PC2", color="target"), size=0.5)
    + gg.theme_minimal()
)

In [ ]:
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

In [ ]:
# X = adata_subset_fix.X
# tsne_ = TSNE(n_components=2, random_state=0, perplexity=100, metric="cosine")
# X_tsne = tsne_.fit_transform(X)

# rep_df = pd.DataFrame(
#     X_tsne, index=adata_subset_fix.obs_names, columns=["t-SNE1", "t-SNE2"]
# ).assign(target=adata_subset_fix.obs["target_"])

# (
#     gg.ggplot(rep_df)
#     + gg.geom_point(gg.aes(x="t-SNE1", y="t-SNE2", color="target"), size=0.5)
#     + gg.theme_minimal()
# )